# Classification des cultures par parcelle, imagerie satellite

Prédit le type de culture d'une parcelle (**palmier à huile, cacao, café, manioc, forêt, autre**)
à partir de séries temporelles Sentinel-2.

## Données

Le notebook tourne **dès maintenant sur un jeu simulé** à la structure réelle, pour que le
pipeline soit validé de bout en bout. Dès que tu as tes vraies données, dépose-les dans
`data/parcelles.parquet` : le notebook les utilisera automatiquement, sans rien changer d'autre.

### Sources réelles vérifiées (août 2026)

| Classe | Source | Lien |
|---|---|---|
| Palmier à huile | Descals et al. 2024, 18 812 points de validation étiquetés + rasters 10 m, CC BY 4.0 | `10.5281/zenodo.13379129` |
| Cacao | COLD-CI, polygones étiquetés cacao / non-cacao, Côte d'Ivoire | arXiv 2606.20767 |
| Cacao (carte) | Kalischek et al., *Nature Food* 2023, carte 10 m CI/Ghana | github.com/D1noFuzi/cocoamapping |
| Cultures annuelles | WorldCereal Reference Data Module (orienté céréales) | esa-worldcereal.org/en/reference-data |

⚠️ `data/` est exclu de git : ni les données personnelles des fermiers, ni les rasters n'y sont commités.

## Format attendu pour tes vraies données

Une ligne par parcelle, dans `data/parcelles.parquet` :

- `classe` : l'étiquette à prédire (une des valeurs de `CLASSES`)
- `bloc` : identifiant de la zone géographique de la parcelle (**indispensable**, voir plus bas)
- `B2_01` … `B12_12` : réflectance moyenne de la parcelle, une colonne par bande et par mois

Le champ `bloc` sert à découper apprentissage / test **spatialement**. Deux parcelles voisines se
ressemblent fortement (même sol, même météo, même date d'image) : avec un découpage aléatoire
classique, des parcelles quasi identiques se retrouvent des deux côtés et le score mesuré est
surestimé. La cellule de comparaison plus bas chiffre cet écart.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

DOSSIER = Path("data")
FICHIER_PARCELLES = DOSSIER / "parcelles.parquet"
FICHIER_MODELE = DOSSIER / "modele_cultures.joblib"

CLASSES = ["palmier_huile", "cacao", "cafe", "manioc", "foret", "autre"]
# Optique Sentinel-2, puis radar Sentinel-1 (VV, VH). Le radar porte la structure de la
# canopée : c'est le seul signal qui sépare palmier et forêt, toutes deux vertes et
# saturées en NDVI toute l'année. Les bandes absentes du fichier sont signalées puis
# ignorées, pour que le notebook tourne aussi sur une extraction purement optique.
BANDES = ["B2", "B3", "B4", "B8", "B11", "B12", "VV", "VH"]
MOIS = list(range(1, 13))

ALEA = np.random.default_rng(42)

## Chargement (ou simulation)

Les profils ci-dessous encodent ce qui distingue réellement ces cultures en télédétection :
les **pérennes à canopée fermée** (palmier, cacao, forêt) gardent un NDVI élevé et quasi constant
toute l'année, alors que le **manioc** suit son cycle cultural avec une forte amplitude saisonnière.
C'est cette amplitude, plus que la réflectance instantanée, qui porte l'essentiel du signal.

In [ ]:
# ndvi : niveau moyen | amplitude : creux-à-pic sur l'année | pic : mois du maximum
# nir / swir : niveaux de réflectance proche-infrarouge et moyen-infrarouge
PROFILS = {
    "palmier_huile": dict(ndvi=0.82, amplitude=0.04, pic=6, nir=0.34, swir=0.17),
    "cacao": dict(ndvi=0.76, amplitude=0.07, pic=8, nir=0.31, swir=0.16),
    "cafe": dict(ndvi=0.71, amplitude=0.11, pic=7, nir=0.30, swir=0.19),
    "manioc": dict(ndvi=0.55, amplitude=0.32, pic=9, nir=0.28, swir=0.24),
    "foret": dict(ndvi=0.87, amplitude=0.03, pic=5, nir=0.36, swir=0.13),
    "autre": dict(ndvi=0.30, amplitude=0.15, pic=8, nir=0.22, swir=0.30),
}


def simuler_parcelles(n_parcelles=1200, n_blocs=30):
    """Jeu synthétique réaliste : chaque bloc est une zone avec sa culture dominante."""
    classes = list(PROFILS)
    # Effet de zone : sol, pluviométrie et dates d'acquisition communs à tout un bloc.
    # C'est lui qui rend les parcelles voisines ressemblantes, donc qui crée la fuite spatiale.
    effet_bloc = ALEA.normal(0, 0.04, size=n_blocs)
    # Une culture domine chaque bloc, mais les autres y restent présentes en minorité, sinon
    # un pli de validation entier peut ne contenir aucune parcelle d'une classe donnée.
    compositions = []
    for numero in range(n_blocs):
        poids = np.full(len(classes), 0.45 / (len(classes) - 1))
        poids[numero % len(classes)] = 0.55
        compositions.append(poids)

    lignes = []
    for _ in range(n_parcelles):
        bloc = int(ALEA.integers(n_blocs))
        classe = str(ALEA.choice(classes, p=compositions[bloc]))
        profil = PROFILS[classe]
        # Effet de zone + variabilité propre à la parcelle (âge des plants, densité).
        decalage = effet_bloc[bloc] + ALEA.normal(0, 0.03)
        enr = {"classe": classe, "bloc": bloc}
        for mois in MOIS:
            saison = np.cos(2 * np.pi * (mois - profil["pic"]) / 12)
            ndvi = np.clip(
                profil["ndvi"] + decalage + profil["amplitude"] * saison / 2
                + ALEA.normal(0, 0.02),
                0.02, 0.95,
            )
            nir = np.clip(profil["nir"] + ALEA.normal(0, 0.03), 0.05, 0.60)
            rouge = nir * (1 - ndvi) / (1 + ndvi)
            swir = np.clip(profil["swir"] + ALEA.normal(0, 0.02), 0.02, 0.50)
            enr[f"B8_{mois:02d}"] = nir
            enr[f"B4_{mois:02d}"] = rouge
            enr[f"B3_{mois:02d}"] = rouge * 1.25 + 0.010
            enr[f"B2_{mois:02d}"] = rouge * 1.10 + 0.015
            enr[f"B11_{mois:02d}"] = swir
            enr[f"B12_{mois:02d}"] = swir * 0.70
        lignes.append(enr)
    return pd.DataFrame(lignes)


if FICHIER_PARCELLES.exists():
    parcelles = pd.read_parquet(FICHIER_PARCELLES)
    print(f"Données réelles : {FICHIER_PARCELLES}")
    manquantes = {"classe", "bloc"} - set(parcelles.columns)
    if manquantes:
        raise ValueError(f"Colonnes manquantes dans le fichier : {sorted(manquantes)}")
else:
    parcelles = simuler_parcelles()
    print(
        "⚠️  DONNÉES SIMULÉES : aucun résultat ci-dessous n'est valide pour le terrain.\n"
        f"    Dépose tes vraies données dans {FICHIER_PARCELLES} pour un entraînement réel."
    )

print(f"\n{len(parcelles)} parcelles, {parcelles['bloc'].nunique()} blocs")
print(parcelles["classe"].value_counts().to_string())

In [ ]:
# Les classes réelles ne sont pas celles du jeu simulé : on les relit dans le fichier.
# Sans ça, la matrice de confusion sort vide et le modèle est sauvegardé avec une liste
# de classes qui ne correspond à rien.
CLASSES = sorted(parcelles["classe"].unique())
print(f"Classes : {CLASSES}")

## Variables explicatives

On ne donne pas les 72 réflectances brutes au modèle : on en extrait des **indices de végétation**
et leurs statistiques annuelles, beaucoup plus stables d'une image à l'autre.

- **NDVI** : vigueur de la végétation
- **NDWI** : teneur en eau du couvert
- **NBR** : structure de la canopée (sépare bien pérennes et annuelles)
- **VV - VH** : écart entre les deux polarisations radar, signature de structure de la canopée.
  C'est la seule variable qui sépare palmier et forêt, à condition que l'extraction Earth Engine
  ait fourni les bandes Sentinel-1.

In [ ]:
def _indice(df, bande_a, bande_b):
    """Indice normalisé (a - b) / (a + b), calculé mois par mois."""
    return pd.DataFrame(
        {
            mois: (df[f"{bande_a}_{mois:02d}"] - df[f"{bande_b}_{mois:02d}"])
            / (df[f"{bande_a}_{mois:02d}"] + df[f"{bande_b}_{mois:02d}"])
            for mois in MOIS
        }
    )


def construire_variables(df):
    variables = pd.DataFrame(index=df.index)
    series = {
        "ndvi": _indice(df, "B8", "B4"),
        "ndwi": _indice(df, "B3", "B8"),
        "nbr": _indice(df, "B8", "B12"),
    }
    for nom, serie in series.items():
        variables[f"{nom}_moy"] = serie.mean(axis=1)
        variables[f"{nom}_ecart_type"] = serie.std(axis=1)
        variables[f"{nom}_min"] = serie.min(axis=1)
        variables[f"{nom}_max"] = serie.max(axis=1)
        # Amplitude saisonnière : le discriminant principal pérennes / annuelles.
        variables[f"{nom}_amplitude"] = variables[f"{nom}_max"] - variables[f"{nom}_min"]
    variables["mois_pic_ndvi"] = series["ndvi"].idxmax(axis=1)

    # Une extraction purement optique n'a ni VV ni VH. On le signale au lieu de les ignorer
    # en silence : sans le radar, palmier et forêt restent inséparables et le score plafonne.
    disponibles = [bande for bande in BANDES if f"{bande}_01" in df.columns]
    absentes = [bande for bande in BANDES if bande not in disponibles]
    if absentes:
        print(f"⚠️  Bandes absentes du fichier, ignorées : {absentes}")

    for bande in disponibles:
        colonnes = [f"{bande}_{mois:02d}" for mois in MOIS]
        variables[f"{bande}_moy"] = df[colonnes].mean(axis=1)
        variables[f"{bande}_ecart_type"] = df[colonnes].std(axis=1)

    # Écart VV - VH : les alignements réguliers d'une palmeraie renvoient un écho radar
    # différent de celui d'une forêt naturelle, là où l'optique ne voit aucune différence.
    if "VV" in disponibles and "VH" in disponibles:
        ecart = pd.DataFrame(
            {mois: df[f"VV_{mois:02d}"] - df[f"VH_{mois:02d}"] for mois in MOIS}
        )
        variables["vv_vh_moy"] = ecart.mean(axis=1)
        variables["vv_vh_ecart_type"] = ecart.std(axis=1)

    return variables


X = construire_variables(parcelles)
y = parcelles["classe"]
blocs = parcelles["bloc"]

print(f"{X.shape[1]} variables construites pour {X.shape[0]} parcelles")
X.head()

## Découpage aléatoire vs spatial

La cellule suivante mesure le même modèle des deux façons. L'écart entre les deux scores est la
part d'optimisme due à la fuite spatiale, c'est la différence entre le chiffre qu'on annonce et
celui qu'on obtiendra sur une nouvelle zone.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import (GroupKFold, StratifiedGroupKFold, StratifiedKFold,
                                     cross_val_score)

# Blocs entiers d'un seul côté du découpage, tout en gardant les six classes
# représentées dans chaque pli.
DECOUPAGE_SPATIAL = StratifiedGroupKFold(5, shuffle=True, random_state=0)

modele_test = RandomForestClassifier(n_estimators=300, random_state=0, n_jobs=-1)

score_aleatoire = cross_val_score(
    modele_test, X, y, cv=StratifiedKFold(5, shuffle=True, random_state=0), scoring="f1_macro"
)
score_spatial = cross_val_score(
    modele_test, X, y, cv=DECOUPAGE_SPATIAL, groups=blocs, scoring="f1_macro"
)

print(f"Découpage aléatoire : F1 = {score_aleatoire.mean():.3f} (± {score_aleatoire.std():.3f})")
print(f"Découpage spatial   : F1 = {score_spatial.mean():.3f} (± {score_spatial.std():.3f})")
print(f"\nOptimisme du découpage aléatoire : {score_aleatoire.mean() - score_spatial.mean():+.3f}")
print("C'est le score spatial qu'il faut retenir et communiquer.")

## Comparaison de modèles

Évaluation en validation croisée spatiale uniquement.

In [ ]:
from sklearn.ensemble import HistGradientBoostingClassifier

candidats = {
    "Forêt aléatoire": RandomForestClassifier(
        n_estimators=400, min_samples_leaf=2, random_state=0, n_jobs=-1
    ),
    "Gradient boosting": HistGradientBoostingClassifier(
        max_iter=300, learning_rate=0.08, random_state=0
    ),
}

resultats = {}
for nom, candidat in candidats.items():
    scores = cross_val_score(candidat, X, y, cv=GroupKFold(5), groups=blocs, scoring="f1_macro")
    resultats[nom] = scores.mean()
    print(f"{nom:20s} F1 = {scores.mean():.3f} (± {scores.std():.3f})")

meilleur_nom = max(resultats, key=resultats.get)
modele = candidats[meilleur_nom]
print(f"\nRetenu : {meilleur_nom}")

## Évaluation détaillée

Blocs d'apprentissage et blocs de test entièrement disjoints.

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay, classification_report

app, test = next(GroupKFold(5).split(X, y, groups=blocs))
modele.fit(X.iloc[app], y.iloc[app])
predictions = modele.predict(X.iloc[test])

print(f"Apprentissage : {len(app)} parcelles / Test : {len(test)} parcelles (blocs disjoints)\n")
print(classification_report(y.iloc[test], predictions, zero_division=0))

figure, axes = plt.subplots(figsize=(7, 6))
ConfusionMatrixDisplay.from_predictions(
    y.iloc[test], predictions, labels=CLASSES, normalize="true",
    values_format=".2f", cmap="Blues", xticks_rotation=45, ax=axes, colorbar=False,
)
axes.set_title("Matrice de confusion (proportion par classe réelle)")
axes.set_xlabel("Prédit")
axes.set_ylabel("Réel")
plt.tight_layout()
plt.show()

## Variables les plus utiles

Importance par permutation, mesurée sur le jeu de test : contrairement à l'importance interne
des arbres, elle ne surévalue pas les variables à forte cardinalité.

In [ ]:
from sklearn.inspection import permutation_importance

importance = permutation_importance(
    modele, X.iloc[test], y.iloc[test],
    n_repeats=10, random_state=0, n_jobs=-1, scoring="f1_macro",
)
classement = (
    pd.Series(importance.importances_mean, index=X.columns)
    .sort_values(ascending=False)
    .head(15)
)

figure, axes = plt.subplots(figsize=(8, 5))
classement.sort_values().plot.barh(ax=axes, color="#2b7a4b")
axes.set_title("15 variables les plus discriminantes")
axes.set_xlabel("Perte de F1 quand la variable est mélangée")
plt.tight_layout()
plt.show()

## Modèle final

Réentraîné sur l'ensemble des parcelles disponibles, puis sauvegardé dans `data/` (hors git).

In [ ]:
import joblib

modele.fit(X, y)
DOSSIER.mkdir(parents=True, exist_ok=True)
joblib.dump(
    {"modele": modele, "variables": list(X.columns), "classes": CLASSES},
    FICHIER_MODELE,
)
print(f"Modèle sauvegardé : {FICHIER_MODELE}")

if not FICHIER_PARCELLES.exists():
    print(
        "\n⚠️  Ce modèle est entraîné sur des données SIMULÉES."
        "\n    Il ne doit pas être utilisé pour classer de vraies parcelles."
    )

## Prochaine étape : brancher les vraies données

1. Télécharger les points de validation palmier (Zenodo `10.5281/zenodo.13379129`,
   `Validation_points_GlobalOP2016-2021.zip`) et les polygones cacao de COLD-CI.
2. Extraire pour chaque point/polygone la série Sentinel-2 mensuelle sur un an
   (Google Earth Engine, ou `rasterio` + `geopandas` en local, ces deux paquets ne sont pas
   encore installés).
3. Écrire le résultat dans `data/parcelles.parquet` au format décrit en haut du notebook,
   en renseignant `bloc` avec une grille spatiale (par exemple des carrés de 10 km).
4. Réexécuter le notebook : il basculera seul sur les vraies données.

Les classes `cafe` et `manioc` n'ont pas de source publique étiquetée pour ta zone. Il faudra
les constituer depuis tes propres relevés terrain : c'est là que la liste de fermiers et leurs
codes de parcelle (`organize_farmers.ipynb`) redeviennent utiles.